# SPE10 Model 2: physical layers and PyVista

The published Darcy case uses layer 36 of Model 2; the 2017 Brinkman figure uses layer 1. This notebook reads the archived, unmodified OPM subsets, checks orientation and renders a native PyVista field. Full-volume acquisition is explicit in `plot_spe10_data.py --download`.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/cc867b645e0395135da2e11dd463f0048382ba8600a629d09c307b3f02d26b4a/21_spe10_pyvista-companion.zip"
COMPANION_SHA256 = "cc867b645e0395135da2e11dd463f0048382ba8600a629d09c307b3f02d26b4a"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/21_spe10_pyvista.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"


In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
from IPython.display import Image, SVG, display
from pymhm.io.reservoir import CartesianCellField
from pymhm.postprocessing.visualization import structured_cell_grid, plot_field
from examples.plot_spe10_data import macro_mesh
from examples.plot_pyvista_layout import horizontal_color_scale
record = json.loads((ROOT / "examples/results/spe10/dataset.json").read_text())
print(record["revision"])
with np.load(ROOT / "examples/results/spe10/layer-36.npz") as data:
    k = data["permeability"].copy()
    phi = data["porosity"].copy()
assert k.shape == (60, 220, 3)
assert np.array_equal(k[..., 0], k[..., 1])
assert k[..., 0].min() == 0.002163 and k[..., 0].max() == 8412.63
field = CartesianCellField(k[..., 0], (20, 10))
assert field([[10, 5]])[0] == k[0, 0, 0]
grid = structured_cell_grid((60,220), cell_data={"ln(Kx / mD)":np.log(k[...,0]), "porosity":phi}, spacing=(20,10))
plotter = plot_field(grid, "ln(Kx / mD)", macro_mesh=macro_mesh(), off_screen=True, cmap="jet", window_size=(700,900), name="spe10-material", show_scalar_bar=False)
plotter.camera.SetWindowCenter(0.0, -0.24)
plotter.show_bounds(show_zaxis=False, show_zlabels=False, xtitle="x [ft]", ytitle="y [ft]",
    font_size=10, color="black", location="outer", ticks="outside", fmt="%.0f")
field_actor = plotter.renderer.actors["spe10-material"]
horizontal_color_scale(plotter, field_actor, "ln(Kx / mD)", scientific=False)
output = ROOT / "build/notebooks/spe10-layer36.png"
output.parent.mkdir(parents=True, exist_ok=True)
plotter.screenshot(output)
plotter.close()
display(Image(filename=str(output)))


The cell data remain piecewise constant. The overlay shows the 66-square macrogrid used by the Darcy article. The source files use numerical permeability in mD and geometry in ft; SI conversion must be explicit. Porosity is geological data, not a coefficient in the stationary incompressible pressure equation.


In [ ]:
for figure in ("volume-and-slice", "layers"):
    display(Image(filename=str(ROOT / "docs/figures/spe10" / f"{figure}.png")))
if HISTORICAL_REPLAY:
    for filename in ("l07-figure-5.png", "l13-figure-18.png"):
        display(Image(filename=str(ROOT / "docs/figures/reservoir-papers" / filename)))
else:
    print("Attributed article rasters are optional historical assets; current material figures are generated from pinned SPE10 data.")


Published references: [Paredes et al., Figure 5](https://doi.org/10.1016/j.cam.2023.115415) and [Araya et al., Figure 18](https://doi.org/10.1016/j.cma.2017.05.027). Source data, layers and checksums are recorded in `examples/results/spe10/dataset.json`. The case page states the exact mesh, boundary conditions, tested refinements and any historical ambiguities.
